# ATML PA2 - Task 5 additional experiment: judge raw-output probe (Kaggle, 1x T4 is enough)

Runs `python -m task5_feedback.additional_experiments --part judge_probe` (GPU) and `--part report` (CPU).
It re-asks the fixed pairwise judge (Qwen2.5-3B-Instruct, 4-bit, same rubric and template) in both A/B
orders, and saves the raw text and the next-token probabilities of A / B / TIE.
It reads the committed Task 5 results (`results/task5_feedback/{gsm,transfer}`), so no input dataset is needed.

**Before running:** Accelerator = *GPU T4 x2* (or x1), Internet = *On*.

Output: `task5_additional_artifacts.zip` (results/task5_feedback/additional + report/task5/additional).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"
BRANCH = "main"
WORKDIR = "/tmp/repo"   # outside /kaggle/working: keeps checkpoints out of the notebook output

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
else:  # an existing clone (e.g. from an earlier run) is synced to the latest pushed commit
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run the probe
Resumable: pairs already in `judge_probe.jsonl` are skipped.

In [ ]:
!python -m task5_feedback.additional_experiments --part lenient
!python -m task5_feedback.additional_experiments --part judge_probe
!python -m task5_feedback.additional_experiments --part report

## 5. Quick look

In [ ]:
from IPython.display import Markdown, display
for t in ["judge_raw_outputs", "judge_position_bias", "lenient_accuracy"]:
    p = f"report/task5/additional/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))

## 6. Package the artifacts

In [ ]:
!cd /tmp/repo && zip -qr /kaggle/working/task5_additional_artifacts.zip results/task5_feedback/additional report/task5/additional
!ls -lh /kaggle/working/